# Notebook 04 — Surge Scenario

100 replications of 180-day simulation.
- Days 0-60: Peacetime
- Days 60-63: Surge (72 hours)
- Days 63-180: Recovery

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.simulation import FleetSimulation
from tqdm import tqdm

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120

HOURS_PER_DAY = 24
print('Ready.')

## Run 100 Replications with Surge

In [ ]:
N_REPS = 100
SIM_DAYS = 180
SURGE_START = 60
SURGE_HOURS = 72

all_mcr_series = []
all_queue_series = []

sim = FleetSimulation(
    fleet_size=12,
    o_level_techs_day=8,
    i_level_techs_day=4,
    d_level_techs=6,
    sortie_interval_hours=48,
)

for i in tqdm(range(N_REPS), desc='Surge replications'):
    result = sim.run(
        simulation_days=SIM_DAYS,
        surge_start_day=SURGE_START,
        surge_duration_hours=SURGE_HOURS,
        random_seed=42 + i,
    )
    kpi_df = result['kpi_dataframe']
    all_mcr_series.append(kpi_df['mcr'].values)
    queue_df = pd.DataFrame(result['queue_history'])
    total_queue = queue_df['o_level_queue'] + queue_df['i_level_queue']
    all_queue_series.append(total_queue.values)

# Align series lengths
min_len = min(len(s) for s in all_mcr_series)
mcr_matrix = np.array([s[:min_len] for s in all_mcr_series])
min_q_len = min(len(s) for s in all_queue_series)
queue_matrix = np.array([s[:min_q_len] for s in all_queue_series])

time_days = np.arange(min_len) / HOURS_PER_DAY
q_time_days = np.arange(min_q_len) / HOURS_PER_DAY

print(f'Complete. {N_REPS} replications, {min_len} time steps each.')

## 1. Mean MCR with 95% CI and Shaded Surge Region

In [ ]:
mean_mcr = mcr_matrix.mean(axis=0) * 100
std_mcr = mcr_matrix.std(axis=0) * 100
ci_low = mean_mcr - 1.96 * std_mcr / np.sqrt(N_REPS)
ci_high = mean_mcr + 1.96 * std_mcr / np.sqrt(N_REPS)

fig, ax = plt.subplots(figsize=(14, 5))

# Surge region
surge_end = SURGE_START + SURGE_HOURS / HOURS_PER_DAY
ax.axvspan(SURGE_START, surge_end, color='red', alpha=0.1, label='Surge period')

# MCR with CI
ax.plot(time_days, mean_mcr, color='#2980b9', linewidth=1.5, label='Mean MCR')
ax.fill_between(time_days, ci_low, ci_high, alpha=0.2, color='#2ecc71', label='95% CI')
ax.axhline(y=75, color='#e74c3c', linestyle='--', alpha=0.7, label='75% threshold')

ax.set_xlabel('Simulation Day', fontsize=12)
ax.set_ylabel('MCR (%)', fontsize=12)
ax.set_title('Mean MCR Over Time with Surge Impact', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(40, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Maintenance Queue Length

In [ ]:
mean_queue = queue_matrix.mean(axis=0)

fig, ax = plt.subplots(figsize=(14, 4))
ax.axvspan(SURGE_START, surge_end, color='red', alpha=0.1)
ax.plot(q_time_days, mean_queue, color='#e67e22', linewidth=1, label='Mean Queue Length')
ax.set_xlabel('Simulation Day', fontsize=12)
ax.set_ylabel('Total Queue Length', fontsize=12)
ax.set_title('Maintenance Queue Backlog During and After Surge', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Surge Impact Summary

In [ ]:
# MCR during surge
surge_start_idx = int(SURGE_START * HOURS_PER_DAY)
surge_end_idx = int((SURGE_START + SURGE_HOURS / HOURS_PER_DAY) * HOURS_PER_DAY)
recovery_end_idx = int((SURGE_START + SURGE_HOURS / HOURS_PER_DAY + 30) * HOURS_PER_DAY)

surge_mcr = mcr_matrix[:, surge_start_idx:min(surge_end_idx, min_len)].mean() * 100
post_surge = mcr_matrix[:, min(surge_end_idx, min_len-1):min(recovery_end_idx, min_len)].mean() * 100

# Recovery time: when does mean MCR return to 75%?
post_mean = mcr_matrix[:, min(surge_end_idx, min_len-1):].mean(axis=0) * 100
recovery_hours = None
for t, m in enumerate(post_mean):
    if m >= 75:
        recovery_hours = t
        break

print('SURGE IMPACT SUMMARY')
print('=' * 50)
print(f'MCR during surge:                {surge_mcr:.1f}%')
print(f'MCR first 30 days post-surge:    {post_surge:.1f}%')
if recovery_hours is not None:
    print(f'Recovery time to 75% MCR:        {recovery_hours} hours ({recovery_hours/HOURS_PER_DAY:.1f} days)')
else:
    print('Recovery time to 75% MCR:        Did not recover within simulation')